In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:15pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:15px;}
</style>
"""))

Chroma Vs Pinecorn
- Chroma : 로컬메모리 DB, 인메모리 DB
- Pinecone : 클라우드 vector DB
    (pinecone console에 api key 생성 -> .env 추가) : PINECONE_API_KEY

# 0. 패키지 설치

In [2]:
%pip install pinecone-client langchain-pinecone

  Using cached wrapt-1.17.2-cp310-cp310-win_amd64.whl.metadata (6.5 kB)
  Using cached py_cpuinfo-9.0.0-py3-none-any.whl.metadata (794 bytes)
   ---------------------------------------- 0.0/587.6 kB ? eta -:--:--
   ---------------------------------------- 587.6/587.6 kB 6.9 MB/s eta 0:00:00
Using cached py_cpuinfo-9.0.0-py3-none-any.whl (22 kB)
Using cached wrapt-1.17.2-cp310-cp310-win_amd64.whl (38 kB)

   -- -------------------------------------  1/19 [wrapt]
   ------ ---------------------------------  3/19 [pinecone-plugin-interface]
   ---------- -----------------------------  5/19 [pytest]
   ---------- -----------------------------  5/19 [pytest]
   ---------- -----------------------------  5/19 [pytest]
   ---------- -----------------------------  5/19 [pytest]
   ---------- -----------------------------  5/19 [pytest]
   ------------ ---------------------------  6/19 [pinecone-plugin-assistant]
   ------------ ---------------------------  6/19 [pinecone-plugin-assistant]
   -

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


# 1. Knowledg Base 구성을 위한 데이터 생성

In [3]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = Docx2txtLoader("./tax_docs/소득세법(법률)(제20615호)(20250701).docx")
splitter = RecursiveCharacterTextSplitter(chunk_size = 1500, chunk_overlap = 200)
document_list = loader.load_and_split(text_splitter=splitter)

In [4]:
len(document_list)

183

In [6]:
# embedding : OpenAI API text-embedding-3-large
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
load_dotenv()
embedding = OpenAIEmbeddings(model = "text-embedding-3-large")

In [8]:
%%time
# pinecone vector database
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

pc = Pinecone()
# 데이터를 처음 저장(업로드)할 때
index_name = "tax-index"
database = PineconeVectorStore.from_documents(
    documents = document_list,
    embedding=embedding,
    index_name=index_name
)

CPU times: total: 7.45 s
Wall time: 14.1 s


# 1-1. PineconeDB에서 Load

In [2]:
# embedding : OpenAI API text-embedding-3-large
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
load_dotenv()
embedding = OpenAIEmbeddings(model = "text-embedding-3-large")

In [5]:
index_name = "tax-index"

In [6]:
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
database = PineconeVectorStore(
    embedding=embedding, # query 임베딩, 유사도 검색
    index_name=index_name
)

# 2. 답변 생성을 위한 Retrieval

In [16]:
query = "연봉 5000만원인 거주자의 소득세는 얼마인가요?"
# retrieved_docs = database.similarity_search_with_score (query, k=10,) # 튜플형태로 유사도 수치를 같이 반환
# print(retrieved_docs[0][0].page_content)
# print(retrieved_docs[0][1])
retrieved_docs = database.similarity_search(query, k=10,)

In [18]:
print(retrieved_docs[0].page_content)

[전문개정 2009. 12. 31.]



제10조(납세지의 변경신고) 거주자나 비거주자는 제6조부터 제9조까지의 규정에 따른 납세지가 변경된 경우 변경된 날부터 15일 이내에 대통령령으로 정하는 바에 따라 그 변경 후의 납세지 관할 세무서장에게 신고하여야 한다.

[전문개정 2009. 12. 31.]



제11조(과세 관할) 소득세는 제6조부터 제10조까지의 규정에 따른 납세지를 관할하는 세무서장 또는 지방국세청장이 과세한다.

[전문개정 2009. 12. 31.]



제2장 거주자의 종합소득 및 퇴직소득에 대한 납세의무 <개정 2009. 12. 31.>



제1절 비과세 <개정 2009. 12. 31.>



제12조(비과세소득) 다음 각 호의 소득에 대해서는 소득세를 과세하지 아니한다. <개정 2010. 12. 27., 2011. 7. 25., 2011. 9. 15., 2012. 2. 1., 2013. 1. 1., 2013. 3. 22., 2014. 1. 1., 2014. 3. 18., 2014. 12. 23., 2015. 12. 15., 2016. 12. 20., 2018. 3. 20., 2018. 12. 31., 2019. 12. 10., 2019. 12. 31., 2020. 6. 9., 2020. 12. 29., 2022. 8. 12., 2022. 12. 31., 2023. 8. 8., 2023. 12. 31., 2024. 12. 31.>

1. 「공익신탁법」에 따른 공익신탁의 이익

2. 사업소득 중 다음 각 목의 어느 하나에 해당하는 소득

가. 논ㆍ밭을 작물 생산에 이용하게 함으로써 발생하는 소득

나. 1개의 주택을 소유하는 자의 주택임대소득(제99조에 따른 기준시가가 12억원을 초과하는 주택 및 국외에 소재하는 주택의 임대소득은 제외한다) 또는 해당 과세기간에 대통령령으로 정하는 총수입금액의 합계액이 2천만원 이하인 자의 주택임대소득(2018년 12월 31일 이전에 끝나는 과세기간까지 발생하는 소득으로 한정한다). 이 경우 주택 수의 

In [28]:
retriever = database.as_retriever(
#     search_kwargs = {"k":4}
)
retriever.invoke(query)

[Document(id='1ffa22e5-ccda-4733-8e14-07d348cf79f3', metadata={'source': './tax_docs/소득세법(법률)(제20615호)(20250701).docx'}, page_content='[전문개정 2009. 12. 31.]\n\n\n\n제10조(납세지의 변경신고) 거주자나 비거주자는 제6조부터 제9조까지의 규정에 따른 납세지가 변경된 경우 변경된 날부터 15일 이내에 대통령령으로 정하는 바에 따라 그 변경 후의 납세지 관할 세무서장에게 신고하여야 한다.\n\n[전문개정 2009. 12. 31.]\n\n\n\n제11조(과세 관할) 소득세는 제6조부터 제10조까지의 규정에 따른 납세지를 관할하는 세무서장 또는 지방국세청장이 과세한다.\n\n[전문개정 2009. 12. 31.]\n\n\n\n제2장 거주자의 종합소득 및 퇴직소득에 대한 납세의무 <개정 2009. 12. 31.>\n\n\n\n제1절 비과세 <개정 2009. 12. 31.>\n\n\n\n제12조(비과세소득) 다음 각 호의 소득에 대해서는 소득세를 과세하지 아니한다. <개정 2010. 12. 27., 2011. 7. 25., 2011. 9. 15., 2012. 2. 1., 2013. 1. 1., 2013. 3. 22., 2014. 1. 1., 2014. 3. 18., 2014. 12. 23., 2015. 12. 15., 2016. 12. 20., 2018. 3. 20., 2018. 12. 31., 2019. 12. 10., 2019. 12. 31., 2020. 6. 9., 2020. 12. 29., 2022. 8. 12., 2022. 12. 31., 2023. 8. 8., 2023. 12. 31., 2024. 12. 31.>\n\n1. 「공익신탁법」에 따른 공익신탁의 이익\n\n2. 사업소득 중 다음 각 목의 어느 하나에 해당하는 소득\n\n가. 논ㆍ밭을 작물 생산에 이용하게 함으로써 발생하는 소득\n\n나. 1개의 주택을 소유하는 자의 주택임대소득(

# 3. 제공되는 prompt를 활용하여 답변 생성

In [20]:
from langchain import hub
prompt = hub.pull("rlm/rag-prompt")

In [22]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model = "gpt-4.1-nano")

In [30]:
from langchain.chains import RetrievalQA
qa_chain = RetrievalQA.from_chain_type(
    llm = llm,
    retriever = retriever,
    chain_type_kwargs={"prompt":prompt},
)

In [31]:
ai_message = qa_chain.invoke({"query":query})

print(ai_message)

{'query': '연봉 5000만원인 거주자의 소득세는 얼마인가요?', 'result': '연봉 5000만원인 거주자의 소득세는 정확한 금액을 알 수 없으나, 일반적으로 종합소득에 대해 세율이 적용되어 약 10~24% 수준의 세금이 부과됩니다. 소득세는 공제, 세율 구간 등 여러 요인에 따라 차이가 있으니 구체적인 계산은 세무 전문가 또는 관련 세법표를 참고해야 합니다. 따라서, 정확한 세액 산출을 위해서는 상세 소득 공제 내역과 세율을 적용해야 합니다.'}
